## Environment Variables for Database Connection

This project uses a `.env` file to securely store database credentials. Make sure to install the `python-dotenv` package to load these variables in your notebook.

In [ ]:
# Importing Necessary Libraries
import pandas as pd
import numpy as np

In [ ]:
# Loading the dataset
yanki_df = pd.read_csv('dataset/rawdata/yanki_ecommerce.csv')
yanki_df.head(5)

In [ ]:
# Getting information about the dataset
yanki_df.info()

In [ ]:
# Drop missing values
yanki_df.dropna(subset=["Order_ID", "Customer_ID"], inplace=True)
# yanki_df.info()

In [ ]:
# Convert Order_Date from string to datetime object
yanki_df["Order_Date"] = pd.to_datetime(yanki_df["Order_Date"], errors="coerce")
yanki_df.info()

In [ ]:
# Data Cleaning and Normalization

# customers table
customers_df = (
    yanki_df[["Customer_ID", "Customer_Name", "Email", "Phone_Number"]]
    .copy()
    .drop_duplicates()
    .reset_index(drop=True)
)

# products table
products_df = (
    yanki_df[["Product_ID", "Product_Name", "Brand", "Category", "Price"]]
    .copy()
    .drop_duplicates()
    .reset_index(drop=True)
)

# Shipping Address table
shipping_address_df = (
    yanki_df[
        ["Customer_ID", "Shipping_Address", "City", "State", "Country", "Postal_Code"]
    ]
    .copy()
    .drop_duplicates()
    .reset_index(drop=True)
)

# Creating shipping address ID
shipping_address_df.index.name = "Shipping_ID"
shipping_address_df.reset_index(inplace=True)

# Orders table
orders_df = (
    yanki_df[
        [
            "Order_ID",
            "Customer_ID",
            "Product_ID",
            "Quantity",
            "Total_Price",
            "Order_Date",
        ]
    ]
    .copy()
    .drop_duplicates()
    .reset_index(drop=True)
)

# Payment method table
payment_method_df = (
    yanki_df[["Order_ID", "Payment_Method", "Transaction_Status"]]
    .copy()
    .drop_duplicates()
    .reset_index(drop=True)
)

# customers_df.head()
# products_df.head()
# shipping_address_df.head()
# orders_df.head()
# payment_method_df.head()

In [ ]:
# Save the cleaned and normalized data to new CSV files

customers_df.to_csv("dataset/cleandata//customers.csv", index=False)
products_df.to_csv("dataset/cleandata//products.csv", index=False)
shipping_address_df.to_csv("dataset/cleandata//shipping_address.csv", index=False)
orders_df.to_csv("dataset/cleandata//orders.csv", index=False)
payment_method_df.to_csv("dataset/cleandata//payment_method.csv", index=False)

In [ ]:
import psycopg2

In [ ]:
from dotenv import load_dotenv
import os
import psycopg2

# Load environment variables from .env file
load_dotenv()

# Get database connection variables from environment
db_host = os.getenv("DB_HOST")
db_name = os.getenv("DB_NAME")
db_user = os.getenv("DB_USER")
db_password = os.getenv("DB_PASSWORD")


def get_db_connection():
    connection = psycopg2.connect(
        host=db_host,
        database=db_name,
        user=db_user,
        password=db_password,
    )
    return connection

In [ ]:
conn = get_db_connection()

In [ ]:
# Create our SQL tables
def create_tables():
    conn = get_db_connection()
    cursor = conn.cursor()

    statements = [
        "CREATE SCHEMA IF NOT EXISTS yanki;",
        "DROP TABLE IF EXISTS yanki.payment_method CASCADE;",
        "DROP TABLE IF EXISTS yanki.orders CASCADE;",
        "DROP TABLE IF EXISTS yanki.shipping_address CASCADE;",
        "DROP TABLE IF EXISTS yanki.products CASCADE;",
        "DROP TABLE IF EXISTS yanki.customers CASCADE;",
        """
        CREATE TABLE IF NOT EXISTS yanki.customers (
            Customer_ID UUID PRIMARY KEY,
            Customer_Name TEXT,
            Email TEXT,
            Phone_Number TEXT
        );
        """,
        """
        CREATE TABLE IF NOT EXISTS yanki.products (
            Product_ID UUID PRIMARY KEY,
            Product_Name TEXT,
            Brand TEXT,
            Category TEXT,
            Price FLOAT
        );
        """,
        """
        CREATE TABLE IF NOT EXISTS yanki.shipping_address (
            shipping_ID INTEGER PRIMARY KEY,
            Customer_ID UUID,
            Shipping_Address TEXT,
            City TEXT,
            State TEXT,
            Country TEXT,
            Postal_Code INTEGER,
            FOREIGN KEY (Customer_ID) REFERENCES yanki.customers(Customer_ID)
        );
        """,
        """
        CREATE TABLE IF NOT EXISTS yanki.orders (
            Order_ID UUID PRIMARY KEY,
            Customer_ID UUID,
            Product_ID UUID,
            Quantity INTEGER,
            Total_Price FLOAT,
            Order_Date DATE,
            FOREIGN KEY (Customer_ID) REFERENCES yanki.customers(Customer_ID),
            FOREIGN KEY (Product_ID) REFERENCES yanki.products(Product_ID)
        );
        """,
        """
        CREATE TABLE IF NOT EXISTS yanki.payment_method (
            Order_ID UUID,
            Payment_Method TEXT,
            Transaction_Status TEXT,
            FOREIGN KEY (Order_ID) REFERENCES yanki.orders(Order_ID)
        );
        """,
    ]

    for sql in statements:
        cursor.execute(sql)

    conn.commit()
    cursor.close()
    conn.close()


create_tables()
print("Schema and tables created successfully.")